In [14]:
import re
import os
import numpy as np
from astropy.io import ascii
from astropy.io import fits
from astropy.table import Table
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
from io import StringIO
from matplotlib.ticker import AutoMinorLocator
from astropy.constants import L_sun
from matplotlib import gridspec
import astropy.units as u
from math import pi

import astropy.units as u
import astropy.cosmology.units as cu
from astropy.cosmology import LambdaCDM, FlatLambdaCDM
from astropy.cosmology import Planck18

In [56]:
def z_tompc(z):
    Z = z * cu.redshift
    d = Z.to(u.Mpc, cu.redshift_distance(Planck18, kind="luminosity"))
    return round(d.value,3)

def z_return(name):
    DF = pd.read_csv('/home/holman/SPS/Final_speclist.csv')
    MJD = int(name[12:17])
    PLATE = int(name[18:22])
    FIBER = int(name[23:26])

    DF_n = DF[ (DF['MJD']==MJD) & (DF['PLATE']==PLATE) & (DF['FIBERID']==FIBER) ]

    if len(DF_n)==1:
        z = DF_n['Z'].iloc[0]
        return z  

def ch_rem(word):
    word = str(word)
    rem = word.replace('[','')
    rem = rem.replace(']','')

    return str(rem)

z_tompc(0.09771754)

464.258

In [45]:
home = os.path.expanduser("~")
spectra_folder = home + '/HIIGs/Tex_spectra/'
spectra_list = os.listdir(spectra_folder)
spectra_list.sort()

lambda_first_values = []

for i in range(len(spectra_list)):
    with open(spectra_folder + spectra_list[i], 'r') as file:
        data = file.read()
        first_lambda = data[0:6]
        lambda_first_values.append(first_lambda)
        file.close()
        del data

In [51]:
n_spec = '10'
base_dir = '/home/holman/BASES/MILESM.Sbmm/'
obs_dir = '/home/holman/HIIGs/Tex_spectra/'
out_dir = '/home/holman/FADOv1b/output/'
plots_dir = '/home/holman/FADOv1b/plots/'

config = '/home/holman/FADOv1b/FADO.config'

units  = 1.e-17

d = 20

res = 2.3

Olsyn_ini = '-'
Olsyn_fin = '9000.0'
Odlsyn = '1.0'

ext_laws = ['ALLR','CALR','CALE','CCMR','FLMC','GOR1','GOR2','GOR3','PREV','SFMW']
BASE = 'Base_MILESM.Sbmm'

In [ ]:
./FADO -i /home/holman/HIIGs/Tex_spectra/[001]spSpec-51608-0267-421.tex -b /home/holman/BASES/MILESM.Sbmm/Base_MILESM.Sbmm -s 3481 9000 1 -r 2.3 -d 530.1 -e CALE -o ./output/001CAL.MILES150 -p ./plots/[001]CAL.MILES150 -u 1.e-17

In [58]:
input_folder = home + '/FADOv1b/inputs/'


for i in range(len(spectra_list)):
    with open(f"{input_folder}/lsFADO_FR{spectra_list[i][0:5]}.txt", 'w') as file:

        n_spec = int(n_spec)

        Distancia = z_tompc(z_return(spectra_list[i]))

        brack_rem = ch_rem(spectra_list[i][0:5])   

        for aux in range(n_spec):
            file.write(f"./FADO -i {obs_dir + spectra_list[i]} -b {base_dir +  BASE} -s {lambda_first_values[i]} {Olsyn_fin} {Odlsyn} -r {res} -d {Distancia} -e {ext_laws[aux]} -o {out_dir + brack_rem + ext_laws[aux]}.MILES150 -p {plots_dir +spectra_list[i][0:5]+ ext_laws[aux]}.MILES150 -u {units} -c {config}\n")
            